In [1]:
# NLP Assignment 5: Token Classification (POS Tagging & Chunking)
### Fine-Tuning BERT using Hugging Face
## Step 1: Install & Import Libraries

!pip install transformers datasets seqeval
!pip install transformers[torch]
!pip install torch
!pip install ipywidgets
!pip install torch ipywidgets

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report

## Step 2: Load Dataset (CoNLL-2003)

dataset = load_dataset("conll2003", revision="refs/convert/parquet")
print(dataset)

## Step 3: Load Tokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

## Step 4: Tokenization & Label Alignment

label_list = dataset["train"].features["chunk_tags"].feature.names

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(example["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## Step 5: Model Setup

model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased',num_labels=len(label_list))

## Step 6: Training

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator 
)

trainer.train()

## Step 7: Evaluation

predictions, labels, _ = trainer.predict(tokenized_datasets['validation'])
preds = np.argmax(predictions, axis=2)

true_labels = [[label_list[l] for l in label if l != -100] for label in labels]
true_preds = [[label_list[p] for (p, l) in zip(pred, label) if l != -100]
              for pred, label in zip(preds, labels)]

print(classification_report(true_labels, true_preds))

## Step 8: Inference

sentence = "John works at Google in California"

tokens = sentence.split()
inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs)
predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)[0]

for token, pred in zip(tokens, predictions):
    print(f"{token} → {label_list[pred]}")

TypeError: must be called with a dataclass type or instance